FASE 1: Carga de librerías necesarias
Descripción

En esta etapa se importan las librerías computacionales necesarias para realizar el procesamiento, análisis y transformación de los microdatos provenientes de la Encuesta Demográfica y de Salud Familiar (ENDES).

Se utilizará principalmente:

Pandas: manipulación y análisis de datos tabulares.
NumPy: operaciones numéricas y creación de variables.
Matplotlib: generación de gráficos exploratorios.

In [ ]:
# =====================================================
# FASE 1: IMPORTACIÓN DE LIBRERÍAS
# =====================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt


print("Librerías cargadas correctamente")

Librerías cargadas correctamente


FASE 2: Carga de microdatos ENDES
Descripción

La ENDES está organizada mediante diferentes módulos.

Para esta primera etapa utilizaremos:

RECH0

Contiene información relacionada con:

identificación del hogar
ubicación geográfica
características generales de la vivienda
RECH6

Contiene información relacionada con:

características del niño
edad
sexo
variables relacionadas con anemia

Los archivos serán cargados utilizando codificación latin-1, debido a que algunas bases ENDES pueden contener caracteres especiales del idioma español.

In [ ]:
# =====================================================
# FASE 2: CARGA DE MÓDULOS ENDES
# =====================================================


# Cargar información del hogar

rech0 = pd.read_csv(
    "RECH0.csv",
    encoding="latin-1",
    low_memory=False
)


# Cargar información del niño

rech6 = pd.read_csv(
    "RECH6.csv",
    encoding="latin-1",
    low_memory=False
)



print("Módulo RECH0 cargado:")
print(rech0.shape)


print("\nMódulo RECH6 cargado:")
print(rech6.shape)

Módulo RECH0 cargado:
(37479, 44)

Módulo RECH6 cargado:
(22741, 41)


FASE 3: Exploración inicial de variables
Descripción

Antes de integrar los módulos, se revisa la estructura de las bases para identificar:

nombres de variables
disponibilidad de información
claves de unión

Esta etapa es importante porque permite documentar qué variables serán utilizadas posteriormente.

In [ ]:
# =====================================================
# FASE 3: REVISIÓN DE VARIABLES
# =====================================================


print("Variables disponibles en RECH0")

print(
    rech0.columns.tolist()
)



print("\nVariables disponibles en RECH6")

print(
    rech6.columns.tolist()
)

Variables disponibles en RECH0
['ID1', 'HHID', 'HV000', 'HV001', 'HV002', 'HV002A', 'HV003', 'HV004', 'HV007', 'HV008', 'HV009', 'HV010', 'HV011', 'HV012', 'HV013', 'HV014', 'HV015', 'HV017', 'HV020', 'HV021', 'HV023', 'HV024', 'HV025', 'HV026', 'HV027', 'HV028', 'HV030', 'HV031', 'HV032', 'HV033', 'HV035', 'HV040', 'HV041', 'HV042', 'HV043', 'HV044', 'UBIGEO', 'HV022', 'CODCCPP', 'NOMCCPP', 'NCONGLOME', 'HV005', 'longitudx', 'latitudy']

Variables disponibles en RECH6
['ï»¿ID1', 'HHID', 'HC0', 'HC1', 'HC2', 'HC3', 'HC4', 'HC5', 'HC6', 'HC7', 'HC8', 'HC9', 'HC10', 'HC11', 'HC12', 'HC13', 'HC15', 'HC16', 'HC19', 'HC27', 'HC30', 'HC31', 'HC32', 'HC33', 'HC51', 'HC52', 'HC53', 'HC55', 'HC56', 'HC57', 'HC58', 'HC60', 'HC61', 'HC62', 'HC63', 'HC64', 'HC68', 'HC70', 'HC71', 'HC72', 'HC73']


FASE 4: Verificación de la variable geográfica
Descripción

Antes de seleccionar la región Puno, debemos verificar la codificación del departamento dentro de la ENDES.

No debemos asumir el código; primero comprobamos la distribución.

In [ ]:
# =====================================================
# FASE 4: REVISIÓN DE DEPARTAMENTOS
# =====================================================


rech0["HV024"].value_counts().sort_index()

,count
HV024,
1,1514
2,1312
3,1343
4,1307
5,1505
6,1383
7,1395
8,1300
9,1425


FASE 5: Integración de módulos ENDES
Descripción

Se unirán los módulos RECH0 y RECH6 mediante la variable:

HHID

Esta variable representa el identificador único del hogar.

La integración permitirá combinar:

información del hogar
información del niño

en una sola base analítica.

In [ ]:
# =====================================================
# FASE 5: INTEGRACIÓN DE BASES
# =====================================================


endes = pd.merge(
    rech0,
    rech6,
    on="HHID",
    how="inner"
)


print(
    "Base ENDES integrada:"
)

print(
    endes.shape
)

Base ENDES integrada:
(22741, 84)


FASE 6: Selección de población de estudio
Descripción

La investigación está enfocada en:

niños residentes en la región Puno
edad entre 6 y 35 meses

Por ello se aplican dos filtros:

Departamento Puno.
Edad del niño.

In [ ]:
# =====================================================
# FASE 6: SELECCIÓN DE POBLACIÓN OBJETIVO
# =====================================================


# Seleccionar Puno

datos_puno = endes[
    endes["HV024"] == 21
].copy()



print(
    "Registros Puno:"
)

print(
    datos_puno.shape
)

Registros Puno:
(657, 84)


In [ ]:
# Selección niños 6 a 35 meses


datos_puno_edad = datos_puno[
    (datos_puno["HC1"] >= 6) &
    (datos_puno["HC1"] <= 35)
].copy()



print(
    "Niños 6-35 meses:"
)

print(
    datos_puno_edad.shape
)

Niños 6-35 meses:
(337, 84)


FASE 7: Creación inicial de variable objetivo
Descripción

Para el modelo de Deep Learning la variable objetivo será binaria:

Código	Significado
1	Presenta anemia
0	No presenta anemia

La variable original ENDES:

HC57

será transformada en:

target_anemia

In [ ]:
# =====================================================
# FASE 7: VARIABLE OBJETIVO
# =====================================================


datos_final = datos_puno_edad.dropna(
    subset=["HC57"]
)


datos_final = datos_final[
    datos_final["HC57"] != 9
]


datos_final["target_anemia"] = np.where(
    datos_final["HC57"].isin([1,2,3]),
    1,
    0
)



print(
    datos_final["target_anemia"].value_counts()
)

target_anemia
1    228
0    100
Name: count, dtype: int64


In [ ]:
# =====================================================
# EXPORTAR DATASET A EXCEL
# =====================================================

nombre_excel = "ENDES_PUNO_ANEMIA_2021.xlsx"

datos_final.to_excel(
    nombre_excel,
    index=False
)

print("Archivo Excel generado correctamente:")
print(nombre_excel)

Archivo Excel generado correctamente:
ENDES_PUNO_ANEMIA_2021.xlsx
